In [ ]:
# This Python code perfectly reproduces the caret::train function in R with the following specifications
# -R code------
# train_control <- caret::trainControl(method = 'cv', number = 10, savePredictions = 'all', seeds = 2024)
# tune_grid <- expand.grid(
#   mtry = 8,                     
#   splitrule = 'extratrees',
#   min.node.size = 49
# )

# rf_full_model <- caret::train(
#   farm_area_ha ~ .,
#   data = lsms_spatial,
#   method = 'ranger',
#   trainControl = train_control,
#   keep.inbag = T,
#   tuneGrid = tune_grid,
#   importance  = 'permutation', # how to get this in Python??
#   metric = 'RMSE',
#   min.bucket = 5,
#   num.trees = 1500
# )

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

import rasterio
import joblib
import time
# ---------------------------------------------------------------------------------------------
# Load table and light data wrangling
lsms_spatial = pd.read_csv('../data/processed/lsms_spatial_with_country_names.csv')     # this is with SPAM 2017 cropland raster according to script 3.1-

# lsms_spatial = lsms_spatial[~ lsms_spatial['country'] .isin (['Ghana', 'Rwanda'])]
lsms_spatial = lsms_spatial[['farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita',
       'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market']]
lsms_spatial = lsms_spatial.dropna()
# lsms_spatial = lsms_spatial[:1000] # uncomment to run script faster
print(lsms_spatial.columns)
print(lsms_spatial)

# define input and output
X = lsms_spatial.drop(columns = ['farm_area_ha'])
y = lsms_spatial['farm_area_ha']
# -----------------------------------------------------------------------------------------------




# Random forest models with and without extra-trees regressor
print('---------------------Without extra-tree regressor------------------------------')
deb = time.time()
# train RF without splitrule = 'extratrees'
rf = RandomForestRegressor(
    n_estimators = 1500, 
    criterion = 'squared_error', 
    min_samples_split = 49,
    min_samples_leaf = 5, 
    max_features = 8, 
    oob_score = True, 
    bootstrap = True, 
    random_state = 2024
)

# Perform cross-validation to get CV R-squared
cv_scores_rf = cross_val_score(rf, X, y, cv = 10, scoring = 'r2', n_jobs = -1)
cv_r2_rf = cv_scores_rf.mean()

# Fit the model to get OOB R-squared
rf.fit(X, y)
oob_r2_rf = rf.oob_score_
fin = time.time()
print(f"RF training time:  {fin - deb} seconds")

print("Simple RF CV R-squared: ", cv_r2_rf)
print("Simple RF OOB R-squared: ", oob_r2_rf)

# Extract variable importance from the Random Forest model
rf_importances = rf.feature_importances_
print(rf_importances)

print('---------------------Extra-tree regressor------------------------------')
start_time = time.time()
# Using extra-trees arguments in the RF
# Define the parameter grid
param_grid = {
    'max_features': [3],       # equivalent to mtry = 3
    'min_samples_split': [50], # equivalent to min.node.size = 50
    'min_samples_leaf': [20],  # equivalent to min.bucket = 20
    'n_estimators': [1500]     # number of trees
}

# Initialize the ExtraTreesRegressor
# etr = ExtraTreesRegressor(criterion = 'squared_error', random_state = 2024)
etr = ExtraTreesRegressor(
    criterion = 'squared_error', 
    oob_score=True, 
    bootstrap=True, 
    random_state = 2024
)

# Step 1: Perform over-all cross-validation (to evaluate the approach, not just for the best hyper-parameterized model)
cv_scores_etr = cross_val_score(etr, X, y, cv = 10, scoring = 'r2', n_jobs = -1)
cv_r2_etr = cv_scores_etr.mean()
print("Overall ExtraTreesRegressor CV R-squared: ", {cv_r2_etr})

# Step 2: Model selection
# Perform grid search
grid_search = GridSearchCV(estimator = etr, param_grid = param_grid, cv = 10, n_jobs = -1, verbose = 2)

# Fit model
grid_search.fit(X, y)
end_time = time.time()
print(f"Extra-trees training time: {end_time - start_time} seconds")

# Get the best model
best_model = grid_search.best_estimator_
print("Best Extra-trese Model:", best_model)

# Get the best hyperparameters
best_params = grid_search.best_params_
print("Best Extra-trees Hyperparameters:", best_params)

# Get the best score
best_score = grid_search.best_score_
print("Best Extra-trees Score:", best_score)

# Get the OOB R square for the best model
oob_rsquare = best_model.oob_score_
print("Extra-trees OOB R2 of best Model:", oob_rsquare)

# # Calculate the cross-validation (CV) R-squared value
# cv_r2 = cross_val_score(best_model, X, y, cv = 10, scoring = 'r2').mean()
# print(f"Extra-trees CV R2 of the best model: {cv_r2}")

# Extract variable importance from the Extra Trees model
etr_importances = best_model.feature_importances_
etr_importance_df = pd.DataFrame({'Variable': X.columns, 'Importance': etr_importances})
print("Variable importance (for extra-trees)")
print(etr_importance_df)
etr_importance_df.to_csv('../output/tables/etr_variable_importance.csv', index=False)

# Save the Extra Trees model as a pkl file
joblib.dump(best_model, '../data/processed/rf_best_model.pkl')

Index(['farm_area_ha', 'cropland', 'cattle', 'pop', 'cropland_per_capita',
       'sand', 'slope', 'temperature', 'rainfall', 'maizeyield', 'market'],
      dtype='object')
        farm_area_ha     cropland        cattle         pop  \
0           0.095855  1562.800049    787.684998   79.315643   
1           2.000000  1562.800049    787.684998   79.315643   
2           0.223116  1562.800049    787.684998   79.315643   
3           3.064620  1562.800049    787.684998   79.315643   
4           0.141745  1562.800049    787.684998   79.315643   
...              ...          ...           ...         ...   
166549      0.457500  4146.000000   1945.512085   54.440788   
166550      0.493900  4146.000000   1945.512085   54.440788   
166551      0.263200  4146.000000   1945.512085   54.440788   
166552      0.170000  4146.000000   1945.512085   54.440788   
166553      0.404858  2234.000000  16306.639648  111.792442   

        cropland_per_capita       sand     slope  temperature     rain

In [ ]:
# --------------------------------------------------------------------------------
# predict raster
# Load the input raster
input_file = '../data/processed/stacked_rasters_africa.tif'
with rasterio.open(input_file) as src:
    input_raster = src.read()  # Read all bands
    profile = src.profile

# Reshape the raster data for prediction
n_bands, height, width = input_raster.shape
input_raster_reshaped = input_raster.reshape(n_bands, -1).T  # Reshape to (n_samples, n_features)

# Filter out rows with NaN values
valid_mask = ~np.isnan(input_raster_reshaped).any(axis=1)
input_raster_valid = input_raster_reshaped[valid_mask]

# Predict using the loaded RF model
rf_output_valid = rf.predict(input_raster_valid)  # RF without extra-trees

# Create an output array and fill with NaNs
rf_output_raster = np.full((height * width,), np.nan)
rf_output_raster[valid_mask] = rf_output_valid
rf_output_raster = rf_output_raster.reshape(height, width)

# Update the profile for the RF output raster
rf_profile = profile.copy()
rf_profile.update(count=1)

# Write the RF output raster
output_rf_file = '../data/processed/Final_noextra_rf_predictions_africa.tif'
with rasterio.open(output_rf_file, 'w', **rf_profile) as dst:
    dst.write(rf_output_raster, 1)

In [ ]:
# --------------------------------------------------------------------------------
# predict raster
# Load the input raster
input_file = '../data/processed/stacked_rasters_africa.tif'
with rasterio.open(input_file) as src:
    input_raster = src.read()  # Read all bands
    profile = src.profile

# Reshape the raster data for prediction
n_bands, height, width = input_raster.shape
input_raster_reshaped = input_raster.reshape(n_bands, -1).T  # Reshape to (n_samples, n_features)

# Filter out rows with NaN values
valid_mask = ~np.isnan(input_raster_reshaped).any(axis=1)
input_raster_valid = input_raster_reshaped[valid_mask]

# Predict using the loaded RF model
rf_output_valid = best_model.predict(input_raster_valid)  # RF with extra-trees

# Create an output array and fill with NaNs
rf_output_raster = np.full((height * width,), np.nan)
rf_output_raster[valid_mask] = rf_output_valid
rf_output_raster = rf_output_raster.reshape(height, width)

# Update the profile for the RF output raster
rf_profile = profile.copy()
rf_profile.update(count=1)

# Write the RF output raster
output_rf_file = '../data/processed/rf_predictions_africa.tif'
with rasterio.open(output_rf_file, 'w', **rf_profile) as dst:
    dst.write(rf_output_raster, 1)